# GammaNet jitter analysis: baseline vs matched top-down bias

Deze notebook analyseert de afbeeldingen in `/home/yentl/pytorch_gammanet/Images_jitter`.

Doel:
1. Gebruik dezelfde C/straight-channelclassificatie uit de eerdere batch-analyse.
2. Meet in `h1_exc` hoe contourrepresentatie verandert als jitter oploopt van 0 tot 60.
3. Vergelijk baseline met een **matched top-down bias**:
   - C-afbeeldingen krijgen bias op C-channels.
   - straight-afbeeldingen krijgen bias op straight-channels.
4. Maak CSV-bestanden en figuren voor maskeractivatie, contour enrichment, layer activation en bias-effect als functie van jitter.

Belangrijk: de bias wordt toegepast in de top-down route (`td_fgru_1`), maar de analyse/uitkomst wordt gemeten in `h1_exc`.


In [ ]:
# ============================================================
# 1. Imports, paths, settings
# ============================================================

import os, re, sys, types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

PROJECT_ROOT = Path("/home/yentl/pytorch_gammanet")
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoint_epoch_40.pt"

# Nieuwe jitter-afbeeldingen
IMAGE_DIR = PROJECT_ROOT / "Images_jitter"

# Bestaande masks, maar notebook kan ook on-the-fly masks maken als ze ontbreken
MASK_DIR = PROJECT_ROOT / "contour_masks"

# Output van deze notebook
OUTPUT_DIR = PROJECT_ROOT / "outputs_jitter_topdown_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Channelclassificatie uit je vorige batch-notebook
CLASSIFICATION_CSV = (
    PROJECT_ROOT /
    "outputs_batch_contour_bias" /
    "csv" /
    "channel_classification" /
    "h1_channel_classes_from_mean_enrichment.csv"
)

INPUT_SIZE = (320, 320)
FORCE_TIMESTEPS = 4       # zet op None als je checkpoint-timesteps wilt gebruiken
BIAS_STRENGTH = 0.15

USE_ABS_ACTIVATION = True
TOP_ACTIVATION_PERCENTILE = 90

ANALYSIS_LAYER = "h1_exc"
BOTTOM_UP_LAYERS = ["h0_exc", "h1_exc", "h2_exc", "h3_exc", "h4_exc"]

CONTOURS = ["C", "straight"]
CONDITIONS = ["baseline", "matched_bias"]

# Voor overlayfiguren; pas aan als je minder/meer wil
JITTERS_TO_PLOT = [0, 10, 20, 30, 40, 50, 60]

for sub in [
    "csv",
    "plots",
    "plots/mask_checks",
    "plots/metric_vs_jitter",
    "plots/delta_vs_jitter",
    "plots/layer_activation_vs_jitter",
    "plots/overlays_by_jitter",
]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

print("IMAGE_DIR:", IMAGE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CLASSIFICATION_CSV:", CLASSIFICATION_CSV)


In [ ]:
# ============================================================
# 2. Model loading
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from gammanet.models.vgg16_gammanet_v2 import VGG16GammaNetV2

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

if "config" in checkpoint and "model" in checkpoint["config"]:
    model_config = checkpoint["config"]["model"]
elif "model_config" in checkpoint:
    model_config = checkpoint["model_config"]
else:
    raise KeyError("Could not find model config in checkpoint.")

model = VGG16GammaNetV2(model_config)

state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", None))
if state_dict is None:
    raise KeyError("Could not find model_state_dict or state_dict in checkpoint.")

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

model.to(DEVICE)
model.eval()

print("Checkpoint model.timesteps:", model.timesteps)

if FORCE_TIMESTEPS is not None:
    model.timesteps = int(FORCE_TIMESTEPS)
    print("Forced model.timesteps:", model.timesteps)

print("Model ready.")


In [ ]:
# ============================================================
# 3. Load C/straight channel classification from previous analysis
# ============================================================

if not CLASSIFICATION_CSV.exists():
    raise FileNotFoundError(
        f"Could not find channel classification CSV:\n{CLASSIFICATION_CSV}\n"
        "Run the previous batch contour-bias notebook first, or update CLASSIFICATION_CSV."
    )

channel_class_df = pd.read_csv(CLASSIFICATION_CSV)

required_cols = {"channel", "C_minus_straight_enrichment", "channel_class"}
missing_cols = required_cols - set(channel_class_df.columns)
if missing_cols:
    raise ValueError(f"Classification CSV misses columns: {missing_cols}")

C_CHANNELS = (
    channel_class_df.loc[
        channel_class_df["C_minus_straight_enrichment"] > 0,
        "channel"
    ]
    .astype(int)
    .tolist()
)

STRAIGHT_CHANNELS = (
    channel_class_df.loc[
        channel_class_df["C_minus_straight_enrichment"] < 0,
        "channel"
    ]
    .astype(int)
    .tolist()
)

print("Loaded channel classification:")
print("C_CHANNELS:", len(C_CHANNELS))
print("STRAIGHT_CHANNELS:", len(STRAIGHT_CHANNELS))

display(channel_class_df.head())


In [ ]:
# ============================================================
# 4. Parse jitter images
# ============================================================

transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
])

def normalize_contour_label(label):
    lower = label.strip().lower()
    if lower in ["c", "ccontour", "c_contour"]:
        return "C"
    if lower in ["straight", "line", "straightline", "straight_line"]:
        return "straight"
    return label

def parse_image_filename(path):
    """
    Expected examples:
        C_high_BL_0_J000_024.png
        straight_high_BL_0_J060_006.png

    Flexible enough as long as:
    contour_contrast_quadrant_position_Jxxx_id.png
    """
    parts = path.stem.split("_")
    if len(parts) < 5:
        return None

    contour_type = normalize_contour_label(parts[0])
    if contour_type not in ["C", "straight"]:
        return None

    # Zoek jitter-part zoals J000
    jitter = None
    jitter_idx = None
    for i, p in enumerate(parts):
        if re.fullmatch(r"J\d+", p):
            jitter = int(p.replace("J", ""))
            jitter_idx = i
            break

    if jitter is None:
        return None

    # Probeer standaardvelden
    contrast = parts[1] if len(parts) > 1 else "unknown"
    quadrant = parts[2] if len(parts) > 2 else "unknown"

    try:
        position = int(parts[3])
    except Exception:
        position = 0

    stimulus_id = parts[jitter_idx + 1] if jitter_idx + 1 < len(parts) else "unknown"

    return {
        "filename": path.name,
        "path": str(path),
        "contour_type": contour_type,
        "contrast": contrast,
        "quadrant": quadrant,
        "position": position,
        "jitter": jitter,
        "stimulus_id": stimulus_id,
    }

def load_image_tensor(path):
    pil_img = Image.open(path).convert("RGB")
    img_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)
    return pil_img, img_tensor

rows = []
for path in sorted(IMAGE_DIR.glob("*.png")):
    info = parse_image_filename(path)
    if info is not None:
        rows.append(info)

jitter_df = pd.DataFrame(rows)

if jitter_df.empty:
    raise RuntimeError(f"No valid jitter images found in {IMAGE_DIR}")

jitter_df = jitter_df.sort_values(["contour_type", "jitter", "filename"]).reset_index(drop=True)

print("Images found:", len(jitter_df))
print(jitter_df["contour_type"].value_counts())
print("Jitter range:", jitter_df["jitter"].min(), "-", jitter_df["jitter"].max())
display(jitter_df.head())
display(jitter_df.tail())

jitter_df.to_csv(OUTPUT_DIR / "csv" / "jitter_image_table.csv", index=False)


In [ ]:
# ============================================================
# 5. Contour mask helper
# ============================================================

# Deze mask-maker is gebaseerd op je make_contour_masks.py.
# Hij gebruikt alleen shape/quadrant/position uit de filename.
# Daardoor werkt hetzelfde masker voor alle jitter levels.

N = 512
LINE_WIDTH = 18

def bezier_curve_position(t, Ps):
    t = np.asarray(t)
    P0, P1, P2, P3 = Ps

    return (
        ((1 - t) ** 3)[:, None] * P0 +
        (3 * ((1 - t) ** 2) * t)[:, None] * P1 +
        (3 * (1 - t) * (t ** 2))[:, None] * P2 +
        (t ** 3)[:, None] * P3
    )

def build_templates():
    Ps = np.array([
        [0.75, 0.9],
        [0.2,  0.9],
        [0.2,  0.1],
        [0.75, 0.1],
    ])

    scale = 0.5
    Ps_scaled = Ps * scale

    curve = bezier_curve_position(np.linspace(0, 1, 200), Ps_scaled)
    bx_base = curve[:, 0][20:-20]
    by_base = curve[:, 1][20:-20]

    x_offset = 0.07
    y_offset = 0.94

    bx_base = bx_base - np.min(bx_base) + x_offset
    by_base = by_base - np.max(by_base) + y_offset

    templates = {}

    for shift_idx in range(4):
        bx = bx_base + 0.1 * shift_idx
        by = by_base.copy()

        templates[("C", shift_idx)] = (bx, by)

        x_center = np.mean(bx)
        bx_mirror = 2 * x_center - bx
        templates[("bC", shift_idx)] = (bx_mirror, by)

        n_points = 5
        change = 0.07
        bx_straight = np.full(n_points, np.mean(bx))
        by_straight = np.linspace(np.min(by) + change, np.max(by) - change, n_points)

        templates[("straight", shift_idx)] = (bx_straight, by_straight)

    return templates

TEMPLATES = build_templates()

def draw_template_mask(shape, position):
    bx, by = TEMPLATES[(shape, position)]

    x = bx * N
    y = by * N

    points = list(zip(x, y))

    mask = Image.new("L", (N, N), 0)
    draw = ImageDraw.Draw(mask)

    if shape == "straight":
        draw.line(points, fill=255, width=LINE_WIDTH)
    else:
        draw.line(points, fill=255, width=LINE_WIDTH, joint="curve")

    return mask

def apply_quadrant_transform(mask, original_shape, quadrant):
    effective_shape = original_shape

    if quadrant == "BL":
        pass

    elif quadrant == "BR":
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        if original_shape == "C":
            effective_shape = "bC"
        elif original_shape == "bC":
            effective_shape = "C"

    elif quadrant == "TL":
        mask = mask.transpose(Image.FLIP_TOP_BOTTOM)

    elif quadrant == "TR":
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
        if original_shape == "C":
            effective_shape = "bC"
        elif original_shape == "bC":
            effective_shape = "C"

    else:
        raise ValueError(f"Unknown quadrant: {quadrant}")

    return mask, effective_shape

def make_mask_from_row(row):
    wanted_shape = row["contour_type"]
    quadrant = row["quadrant"]
    position = int(row["position"])

    candidate_masks = []

    if wanted_shape == "straight":
        base_shapes = ["straight"]
    else:
        base_shapes = ["C", "bC"]

    for base_shape in base_shapes:
        mask = draw_template_mask(base_shape, position)
        mask, effective_shape = apply_quadrant_transform(
            mask,
            original_shape=base_shape,
            quadrant=quadrant,
        )

        if effective_shape == wanted_shape:
            candidate_masks.append(mask)

    if len(candidate_masks) == 0:
        raise RuntimeError(f"No candidate mask for {row['filename']}")

    final = Image.new("L", (N, N), 0)

    for mask in candidate_masks:
        final = Image.fromarray(
            np.maximum(np.asarray(final), np.asarray(mask)).astype(np.uint8)
        )

    return final

def load_true_contour_mask(row, target_size):
    """
    Load saved mask if it exists; otherwise build it from filename information.
    Returns boolean mask resized to target_size.
    """
    mask_path = MASK_DIR / row["filename"]

    if mask_path.exists():
        mask = Image.open(mask_path).convert("L")
    else:
        mask = make_mask_from_row(row)

    mask = mask.resize(target_size, resample=Image.NEAREST)
    return np.asarray(mask) > 0


In [ ]:
# ============================================================
# 5B. Visual mask sanity check
# ============================================================

MASK_CHECK_DIR = OUTPUT_DIR / "plots" / "mask_checks"
MASK_CHECK_DIR.mkdir(parents=True, exist_ok=True)

for contour in CONTOURS:
    row = jitter_df[jitter_df["contour_type"] == contour].sort_values("jitter").iloc[0]
    pil_img, _ = load_image_tensor(row["path"])
    mask = load_true_contour_mask(row, target_size=pil_img.size)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(pil_img)
    axes[0].set_title("Stimulus")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Contour mask")
    axes[1].axis("off")

    axes[2].imshow(pil_img)
    axes[2].imshow(mask, cmap="Reds", alpha=0.45)
    axes[2].set_title("Mask overlay")
    axes[2].axis("off")

    plt.suptitle(row["filename"])
    plt.tight_layout()

    save_path = MASK_CHECK_DIR / f"mask_check_{contour}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show()
    plt.close(fig)


In [ ]:
# ============================================================
# 6. Forward helpers and activation extraction
# ============================================================

def reset_and_forward(model, img_tensor):
    model.reset_hidden_states()
    with torch.no_grad():
        return model(img_tensor)

def get_state(model, layer_name):
    state = getattr(model, layer_name, None)
    if state is None:
        raise ValueError(f"{layer_name} is None. Run model first or check layer name.")
    return state

def resize_map_to_image(fmap, pil_img):
    fmap_t = torch.tensor(fmap, dtype=torch.float32)[None, None]
    resized = F.interpolate(
        fmap_t,
        size=pil_img.size[::-1],
        mode="bilinear",
        align_corners=False,
    )
    return resized[0, 0].numpy()

def normalize_for_plot(x, eps=1e-8):
    x = np.asarray(x)
    x = x - np.nanmin(x)
    return x / (np.nanmax(x) + eps)

def all_channel_map(state, use_abs=True, channels=None):
    x = state.detach()
    if channels is not None:
        idx = torch.tensor([int(c) for c in channels], device=x.device, dtype=torch.long)
        x = x[:, idx, :, :]
    if use_abs:
        x = x.abs()
    return x.mean(dim=(0, 1)).detach().cpu().numpy()

def single_channel_map(state, channel):
    x = state.detach().cpu()
    return x[0, int(channel)].numpy()


In [ ]:
# ============================================================
# 7. Top-down bias context manager
# ============================================================

class TDH1Bias:
    """
    Non-invasive bias op td_fgru_1-output.
    De checkpoint op disk wordt niet aangepast.
    """
    def __init__(self, model, channels, strength=0.15, mode="add_mean_abs"):
        self.model = model
        self.channels = [int(c) for c in channels]
        self.strength = float(strength)
        self.mode = mode
        self.handle = None

    def _hook(self, module, inputs, output):
        if not isinstance(output, tuple):
            return output

        exc = output[0]

        if exc is None or len(self.channels) == 0:
            return output

        exc = exc.clone()
        idx = torch.tensor(self.channels, device=exc.device, dtype=torch.long)

        if self.mode == "add_mean_abs":
            scale = exc.detach().abs().mean(dim=(2, 3), keepdim=True) + 1e-8
            exc[:, idx, :, :] = exc[:, idx, :, :] + self.strength * scale[:, idx, :, :]
        elif self.mode == "multiply":
            exc[:, idx, :, :] = exc[:, idx, :, :] * (1.0 + self.strength)
        else:
            raise ValueError(f"Unknown mode: {self.mode}")

        return (exc,) + output[1:]

    def __enter__(self):
        self.handle = self.model.td_fgru_1.register_forward_hook(self._hook)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self.handle is not None:
            self.handle.remove()
        return False

def get_matched_bias_channels(contour_type):
    if contour_type == "C":
        return C_CHANNELS
    if contour_type == "straight":
        return STRAIGHT_CHANNELS
    raise ValueError(f"Unknown contour_type: {contour_type}")

def forward_with_condition(model, img_tensor, row, condition):
    if condition == "baseline":
        return reset_and_forward(model, img_tensor)

    if condition == "matched_bias":
        bias_channels = get_matched_bias_channels(row["contour_type"])
        with TDH1Bias(
            model=model,
            channels=bias_channels,
            strength=BIAS_STRENGTH,
            mode="add_mean_abs",
        ):
            return reset_and_forward(model, img_tensor)

    raise ValueError(f"Unknown condition: {condition}")


In [ ]:
# ============================================================
# 8. Quantification helpers
# ============================================================

def compute_mask_metrics_from_activation(act, contour_mask):
    act = np.asarray(act)

    if USE_ABS_ACTIVATION:
        act = np.abs(act)
    else:
        act = act.copy()

    act = np.nan_to_num(act, nan=0.0, posinf=0.0, neginf=0.0)
    act = act - act.min() + 1e-8

    background_mask = ~contour_mask

    contour_values = act[contour_mask]
    background_values = act[background_mask]

    contour_mean = contour_values.mean()
    background_mean = background_values.mean()

    contour_sum = contour_values.sum()
    total_sum = act.sum()

    contour_area_pct = 100 * contour_mask.mean()

    contour_preference_ratio = contour_mean / (background_mean + 1e-8)
    activation_on_contour_pct = 100 * contour_sum / (total_sum + 1e-8)
    contour_enrichment = activation_on_contour_pct / (contour_area_pct + 1e-8)

    act_threshold = np.percentile(act, TOP_ACTIVATION_PERCENTILE)
    top_activation_mask = act >= act_threshold

    intersection = np.logical_and(contour_mask, top_activation_mask).sum()

    dice_top_activation = (
        2 * intersection /
        (contour_mask.sum() + top_activation_mask.sum() + 1e-8)
    )

    return {
        "contour_mean_activation": contour_mean,
        "background_mean_activation": background_mean,
        "contour_preference_ratio": contour_preference_ratio,
        "activation_on_contour_pct": activation_on_contour_pct,
        "contour_area_pct": contour_area_pct,
        "contour_enrichment": contour_enrichment,
        "dice_top_activation": dice_top_activation,
        "top_activation_percentile": TOP_ACTIVATION_PERCENTILE,
    }

def quantify_groups_for_current_state(model, row, pil_img, layer_name=ANALYSIS_LAYER):
    state = get_state(model, layer_name).detach().cpu()
    contour_mask = load_true_contour_mask(row, target_size=pil_img.size)

    groups = {
        "all_channels": None,
        "C_channels": C_CHANNELS,
        "straight_channels": STRAIGHT_CHANNELS,
        "matched_channels": get_matched_bias_channels(row["contour_type"]),
    }

    out_rows = []

    for group_name, channels in groups.items():
        fmap = all_channel_map(state, use_abs=USE_ABS_ACTIVATION, channels=channels)
        fmap_resized = resize_map_to_image(fmap, pil_img)

        metrics = compute_mask_metrics_from_activation(fmap_resized, contour_mask)

        out_rows.append({
            "filename": row["filename"],
            "contour_type": row["contour_type"],
            "contrast": row["contrast"],
            "quadrant": row["quadrant"],
            "position": row["position"],
            "stimulus_id": row["stimulus_id"],
            "jitter": int(row["jitter"]),
            "layer": layer_name,
            "channel_group": group_name,
            "n_channels": state.shape[1] if channels is None else len(channels),
            **metrics,
        })

    return out_rows

def collect_layer_activation_stats(model, row, condition):
    rows = []
    for layer in BOTTOM_UP_LAYERS:
        state = get_state(model, layer).detach().cpu()
        rows.append({
            "filename": row["filename"],
            "contour_type": row["contour_type"],
            "jitter": int(row["jitter"]),
            "condition": condition,
            "layer": layer,
            "mean_activation": state.mean().item(),
            "mean_abs_activation": state.abs().mean().item(),
            "std_activation": state.std().item(),
            "min_activation": state.min().item(),
            "max_activation": state.max().item(),
        })
    return rows


In [ ]:
# ============================================================
# 9. Run jitter analysis: baseline vs matched top-down bias
# ============================================================

metric_rows = []
layer_stat_rows = []

for _, row in tqdm(jitter_df.iterrows(), total=len(jitter_df)):

    pil_img, img_tensor = load_image_tensor(row["path"])

    for condition in CONDITIONS:

        forward_with_condition(
            model=model,
            img_tensor=img_tensor,
            row=row,
            condition=condition,
        )

        # Mask metrics in h1_exc
        rows = quantify_groups_for_current_state(
            model=model,
            row=row,
            pil_img=pil_img,
            layer_name=ANALYSIS_LAYER,
        )

        for r in rows:
            r["condition"] = condition
            r["bias_strength"] = 0.0 if condition == "baseline" else BIAS_STRENGTH

        metric_rows.extend(rows)

        # Mean activation statistics for all bottom-up layers
        layer_stat_rows.extend(
            collect_layer_activation_stats(
                model=model,
                row=row,
                condition=condition,
            )
        )

metrics_df = pd.DataFrame(metric_rows)
layer_stats_df = pd.DataFrame(layer_stat_rows)

metrics_path = OUTPUT_DIR / "csv" / "jitter_mask_metrics_baseline_vs_matched_bias.csv"
layer_stats_path = OUTPUT_DIR / "csv" / "jitter_layer_activation_stats.csv"

metrics_df.to_csv(metrics_path, index=False)
layer_stats_df.to_csv(layer_stats_path, index=False)

print("Saved:", metrics_path)
print("Saved:", layer_stats_path)

display(metrics_df.head())
display(layer_stats_df.head())


In [ ]:
# ============================================================
# 10. Delta: matched bias - baseline per jitter
# ============================================================

baseline = (
    metrics_df[metrics_df["condition"] == "baseline"]
    .drop(columns=["condition", "bias_strength"])
    .rename(columns={
        "contour_mean_activation": "baseline_contour_mean_activation",
        "background_mean_activation": "baseline_background_mean_activation",
        "contour_preference_ratio": "baseline_contour_preference_ratio",
        "activation_on_contour_pct": "baseline_activation_on_contour_pct",
        "contour_enrichment": "baseline_contour_enrichment",
        "dice_top_activation": "baseline_dice_top_activation",
    })
)

biased = metrics_df[metrics_df["condition"] == "matched_bias"].copy()

merge_cols = [
    "filename",
    "contour_type",
    "contrast",
    "quadrant",
    "position",
    "stimulus_id",
    "jitter",
    "layer",
    "channel_group",
    "n_channels",
    "contour_area_pct",
    "top_activation_percentile",
]

delta_df = biased.merge(
    baseline[
        merge_cols + [
            "baseline_contour_mean_activation",
            "baseline_background_mean_activation",
            "baseline_contour_preference_ratio",
            "baseline_activation_on_contour_pct",
            "baseline_contour_enrichment",
            "baseline_dice_top_activation",
        ]
    ],
    on=merge_cols,
    how="left",
)

delta_df["delta_contour_mean_activation"] = (
    delta_df["contour_mean_activation"] -
    delta_df["baseline_contour_mean_activation"]
)

delta_df["delta_background_mean_activation"] = (
    delta_df["background_mean_activation"] -
    delta_df["baseline_background_mean_activation"]
)

delta_df["delta_contour_preference_ratio"] = (
    delta_df["contour_preference_ratio"] -
    delta_df["baseline_contour_preference_ratio"]
)

delta_df["delta_activation_on_contour_pct"] = (
    delta_df["activation_on_contour_pct"] -
    delta_df["baseline_activation_on_contour_pct"]
)

delta_df["delta_contour_enrichment"] = (
    delta_df["contour_enrichment"] -
    delta_df["baseline_contour_enrichment"]
)

delta_df["delta_dice_top_activation"] = (
    delta_df["dice_top_activation"] -
    delta_df["baseline_dice_top_activation"]
)

delta_path = OUTPUT_DIR / "csv" / "jitter_delta_matched_bias_minus_baseline.csv"
delta_df.to_csv(delta_path, index=False)

print("Saved:", delta_path)
display(delta_df.head())


In [ ]:
# ============================================================
# 11. Plot metrics vs jitter
# ============================================================

METRIC_PLOT_DIR = OUTPUT_DIR / "plots" / "metric_vs_jitter"
METRIC_PLOT_DIR.mkdir(parents=True, exist_ok=True)

metrics_to_plot = [
    ("contour_mean_activation", "Contour mean activation"),
    ("background_mean_activation", "Background mean activation"),
    ("contour_preference_ratio", "Contour preference ratio"),
    ("activation_on_contour_pct", "Activation on contour (%)"),
    ("contour_enrichment", "Contour enrichment"),
    ("dice_top_activation", "Dice top activation"),
]

groups_to_plot = ["all_channels", "matched_channels"]

for metric, ylabel in metrics_to_plot:
    for contour in CONTOURS:
        for group in groups_to_plot:

            sub = metrics_df[
                (metrics_df["contour_type"] == contour) &
                (metrics_df["channel_group"] == group)
            ].sort_values("jitter")

            if sub.empty:
                continue

            fig, ax = plt.subplots(figsize=(8, 5))

            for condition in CONDITIONS:
                csub = sub[sub["condition"] == condition].sort_values("jitter")
                ax.plot(
                    csub["jitter"],
                    csub[metric],
                    marker="o",
                    linewidth=2,
                    label=condition,
                )

            ax.set_xlabel("Jitter")
            ax.set_ylabel(ylabel)
            ax.set_title(f"{ylabel} vs jitter | {contour} | {group}")
            ax.legend()
            ax.grid(alpha=0.3)

            plt.tight_layout()

            save_path = METRIC_PLOT_DIR / f"{metric}_{contour}_{group}.png"
            fig.savefig(save_path, dpi=150, bbox_inches="tight")

            print("Saved:", save_path)
            plt.show()
            plt.close(fig)


In [ ]:
# ============================================================
# 12. Plot matched-bias delta vs jitter
# ============================================================

DELTA_PLOT_DIR = OUTPUT_DIR / "plots" / "delta_vs_jitter"
DELTA_PLOT_DIR.mkdir(parents=True, exist_ok=True)

delta_metrics_to_plot = [
    ("delta_contour_mean_activation", "Δ contour mean activation"),
    ("delta_contour_preference_ratio", "Δ contour preference ratio"),
    ("delta_activation_on_contour_pct", "Δ activation on contour (%)"),
    ("delta_contour_enrichment", "Δ contour enrichment"),
    ("delta_dice_top_activation", "Δ dice top activation"),
]

for metric, ylabel in delta_metrics_to_plot:
    for group in groups_to_plot:

        fig, ax = plt.subplots(figsize=(8, 5))

        for contour in CONTOURS:
            sub = delta_df[
                (delta_df["contour_type"] == contour) &
                (delta_df["channel_group"] == group)
            ].sort_values("jitter")

            if sub.empty:
                continue

            ax.plot(
                sub["jitter"],
                sub[metric],
                marker="o",
                linewidth=2,
                label=contour,
            )

        ax.axhline(0, linestyle="--", linewidth=1)
        ax.set_xlabel("Jitter")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{ylabel} vs jitter | matched bias - baseline | {group}")
        ax.legend()
        ax.grid(alpha=0.3)

        plt.tight_layout()

        save_path = DELTA_PLOT_DIR / f"{metric}_{group}.png"
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

        print("Saved:", save_path)
        plt.show()
        plt.close(fig)


In [ ]:
# ============================================================
# 13. Layer activation vs jitter
# ============================================================

LAYER_PLOT_DIR = OUTPUT_DIR / "plots" / "layer_activation_vs_jitter"
LAYER_PLOT_DIR.mkdir(parents=True, exist_ok=True)

for contour in CONTOURS:
    for layer in BOTTOM_UP_LAYERS:

        sub = layer_stats_df[
            (layer_stats_df["contour_type"] == contour) &
            (layer_stats_df["layer"] == layer)
        ].sort_values("jitter")

        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(8, 5))

        for condition in CONDITIONS:
            csub = sub[sub["condition"] == condition].sort_values("jitter")

            ax.plot(
                csub["jitter"],
                csub["mean_abs_activation"],
                marker="o",
                linewidth=2,
                label=condition,
            )

        ax.set_xlabel("Jitter")
        ax.set_ylabel("Mean abs activation")
        ax.set_title(f"Mean abs activation vs jitter | {contour} | {layer}")
        ax.legend()
        ax.grid(alpha=0.3)

        plt.tight_layout()

        save_path = LAYER_PLOT_DIR / f"mean_abs_activation_{contour}_{layer}.png"
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

        print("Saved:", save_path)
        plt.show()
        plt.close(fig)


In [ ]:
# ============================================================
# 14. Overlay plots for selected jitter levels
# ============================================================

OVERLAY_DIR = OUTPUT_DIR / "plots" / "overlays_by_jitter"
OVERLAY_DIR.mkdir(parents=True, exist_ok=True)

def plot_overlay_for_row_condition(row, condition, channel_group="matched_channels", alpha=0.55):
    pil_img, img_tensor = load_image_tensor(row["path"])

    forward_with_condition(
        model=model,
        img_tensor=img_tensor,
        row=row,
        condition=condition,
    )

    state = get_state(model, ANALYSIS_LAYER).detach().cpu()

    if channel_group == "all_channels":
        channels = None
    elif channel_group == "matched_channels":
        channels = get_matched_bias_channels(row["contour_type"])
    elif channel_group == "C_channels":
        channels = C_CHANNELS
    elif channel_group == "straight_channels":
        channels = STRAIGHT_CHANNELS
    else:
        raise ValueError(channel_group)

    fmap = all_channel_map(
        state,
        use_abs=USE_ABS_ACTIVATION,
        channels=channels,
    )
    fmap_resized = resize_map_to_image(fmap, pil_img)
    fmap_norm = normalize_for_plot(fmap_resized)

    mask = load_true_contour_mask(row, target_size=pil_img.size)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(pil_img)
    axes[0].set_title("Stimulus")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Contour mask")
    axes[1].axis("off")

    axes[2].imshow(fmap_norm, cmap="inferno")
    axes[2].set_title(f"{ANALYSIS_LAYER}\n{channel_group}")
    axes[2].axis("off")

    axes[3].imshow(pil_img)
    axes[3].imshow(fmap_norm, cmap="inferno", alpha=alpha)
    axes[3].imshow(mask, cmap="Reds", alpha=0.20)
    axes[3].set_title("Activation + mask overlay")
    axes[3].axis("off")

    title = (
        f"{row['contour_type']} | jitter={row['jitter']} | "
        f"{condition} | {channel_group} | {row['filename']}"
    )
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()

    save_dir = OVERLAY_DIR / condition / row["contour_type"] / channel_group
    save_dir.mkdir(parents=True, exist_ok=True)

    save_path = save_dir / (
        f"overlay_{row['contour_type']}_J{int(row['jitter']):03d}_"
        f"{condition}_{channel_group}.png"
    )
    fig.savefig(save_path, dpi=150, bbox_inches="tight")

    print("Saved:", save_path)
    plt.show()
    plt.close(fig)

for contour in CONTOURS:
    for jitter in JITTERS_TO_PLOT:
        candidates = jitter_df[
            (jitter_df["contour_type"] == contour) &
            (jitter_df["jitter"] == jitter)
        ]
        if candidates.empty:
            print(f"Skipping {contour} J{jitter:03d}: no image")
            continue

        row = candidates.iloc[0]

        for condition in CONDITIONS:
            plot_overlay_for_row_condition(
                row=row,
                condition=condition,
                channel_group="matched_channels",
                alpha=0.55,
            )


In [ ]:
# ============================================================
# 15. Compact summary tables
# ============================================================

summary_df = (
    metrics_df
    .groupby(["condition", "contour_type", "channel_group"], as_index=False)
    .agg(
        n_images=("filename", "nunique"),
        mean_contour_mean_activation=("contour_mean_activation", "mean"),
        mean_background_mean_activation=("background_mean_activation", "mean"),
        mean_contour_preference_ratio=("contour_preference_ratio", "mean"),
        mean_activation_on_contour_pct=("activation_on_contour_pct", "mean"),
        mean_contour_enrichment=("contour_enrichment", "mean"),
        mean_dice_top_activation=("dice_top_activation", "mean"),
    )
)

delta_summary_df = (
    delta_df
    .groupby(["contour_type", "channel_group"], as_index=False)
    .agg(
        n_images=("filename", "nunique"),
        mean_delta_contour_mean_activation=("delta_contour_mean_activation", "mean"),
        mean_delta_contour_preference_ratio=("delta_contour_preference_ratio", "mean"),
        mean_delta_activation_on_contour_pct=("delta_activation_on_contour_pct", "mean"),
        mean_delta_contour_enrichment=("delta_contour_enrichment", "mean"),
        mean_delta_dice_top_activation=("delta_dice_top_activation", "mean"),
    )
)

summary_df.to_csv(OUTPUT_DIR / "csv" / "jitter_summary_overall.csv", index=False)
delta_summary_df.to_csv(OUTPUT_DIR / "csv" / "jitter_delta_summary_overall.csv", index=False)

display(summary_df)
display(delta_summary_df)


## Interpretatie

De belangrijkste figuren zitten in:

- `plots/metric_vs_jitter/`  
  Baseline versus matched top-down bias als functie van jitter.

- `plots/delta_vs_jitter/`  
  Direct effect van top-down bias: matched bias minus baseline.

- `plots/overlays_by_jitter/`  
  Visuele controle: stimulus, masker, activatiemap en overlay voor geselecteerde jitters.

Voor je hypothese is vooral `contour_enrichment` relevant:
- Als contourrepresentatie slechter wordt bij hogere jitter, verwacht je een dalende baseline-lijn.
- Als top-down modulatie helpt, verwacht je dat matched bias bij hoge jitter minder snel daalt of een positieve delta behoudt.
